# 02 - Biomedical Knowledge Graph Construction & Multi-Hop Querying

**Data Analytics Capstone - Research Pipeline**  
**Author:** Suddhasatwa Bhaumik  
**Institution:** Walsh College (QM640)  

---

### Overview
This notebook covers the construction and multi-hop traversal of the **Biomedical Knowledge Graph (KG)**:
1. **Schema Definition**: Initializes the graph nodes (`graph_nodes`) and relationship edges (`graph_edges`) tables.
2. **Concept & Relationship Ingestion**: Ingests UMLS entities and links co-occurring clinical entities (e.g. `(Disease)-[:ASSOCIATED_WITH]->(Drug)`).
3. **Multi-Hop Subgraph Retrieval**: Executes 2-hop neighborhood traversals to retrieve structured relational context for clinical reasoning.
4. **Graph-of-Thought (GoT) Path Serialization**: Formats subgraphs into logical path strings for LLM injection.


In [ ]:
import os
import json
import logging
import pandas as pd

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Ensure working directory is the repository root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Current Working Directory: {os.getcwd()}")


## 1. Initialize Graph Database Client
We connect to the graph backend and verify schema constraints and tables.


In [ ]:
from src.graph_builder import GraphBuilder

project_id = os.environ.get("GCP_PROJECT", "suddhasatwa-data-projects")
builder = GraphBuilder(project_id=project_id)
builder.connect()
builder.create_constraints_and_indexes()
print("Graph database connection and schema constraints verified successfully.")


## 2. Ingest Concept Nodes and Medical Relationships
We load the preprocessed UMLS entities from `data/processed/processed_data.csv` and ingest them into the graph store.


In [ ]:
processed_csv = "data/processed/processed_data.csv"
df = pd.read_csv(processed_csv)

all_notes_entities = []
for _, row in df.iterrows():
    if pd.notna(row['entities']):
        try:
            entities = json.loads(row['entities'])
            all_notes_entities.append(entities)
        except Exception as e:
            print(f"Parsing error: {e}")

print(f"Loaded entity annotations across {len(all_notes_entities)} admission notes.")
builder.ingest_bulk_entities(all_notes_entities)
print("Bulk ingestion of concept nodes and relationship edges complete.")


## 3. Multi-Hop Subgraph Traversal
We query 2-hop logical relationships centered around a target clinical concept (e.g. Asthma: `C0004096` or Myocardial Infarction: `C0027051`).


In [ ]:
target_cui = "C0004096" # Asthma
subgraph = builder.get_2_hop_subgraph(cui=target_cui)

print(f"Found {len(subgraph)} multi-hop relational paths connected to CUI {target_cui}:")
for path in subgraph[:10]:
    nodes = path.get("nodes", [])
    relations = path.get("relations", [])
    path_str = ""
    for i in range(len(nodes)):
        path_str += f"({nodes[i]})"
        if i < len(relations):
            path_str += f" -[:{relations[i]}]-> "
    print(f"  • {path_str}")


## 4. Graph-of-Thought (GoT) Path Serialization
Format extracted subgraphs into clean textual descriptions ready for injection into LLM prompts.


In [ ]:
from src.rag_engine import RAGEngine

rag_engine = RAGEngine()
formatted_paths = rag_engine.format_graph_paths(subgraph)

print("--- Formatted Graph Context for Prompt Injection ---")
print(formatted_paths)
builder.close()
